# Worked Example: First PSD and FOOOF Run

## Goal
Use the spectral builder for PSD, then run FOOOF on a small epoch subset.

## Expected input files
- `../../data/sample_feedback_start-epo.fif`

In [ ]:
from pathlib import Path
from LFPAnalysis import build_spectral_pipeline_config, run_pipeline

config = build_spectral_pipeline_config(
    Path('../../data/sample_feedback_start-epo.fif'),
    file_format='mne',
    spectral_method='psd',
    baseline_mode='none',
)
result = run_pipeline(config)
print(result.spectral['method'])
print(result.spectral['spectrum'])

## Inspect the result

The stable API currently wraps PSD directly. FOOOF still uses the advanced analysis helper on top of epoched data.

## Utility stack note

This handoff is intentional: `load_lfp(...)` or `run_pipeline(...)` should prepare the MNE object, and `analysis_utils` should take over only for the spectral step that is still advanced. See `11_advanced_utility_interoperability` in the book if you also need custom baselining, surrogates, or downstream statistics.



In [ ]:
from LFPAnalysis import analysis_utils, load_lfp, LoadConfig

epochs = load_lfp(LoadConfig(path=Path('../../data/sample_feedback_start-epo.fif'), file_format='mne'))
epochs = epochs.copy().pick(epochs.ch_names[:4])[:10]
fooof_group, fooof_table = analysis_utils.FOOOF_compute_epochs(
    epochs,
    tmin=float(epochs.times[0]),
    tmax=float(epochs.times[-1]),
    peak_width_limits=(1, 12),
    min_peak_height=0.0,
    peak_threshold=2.0,
    max_n_peaks=4,
    freq_range=(2, 40),
)
print(fooof_table.head())

## Meaningful parameter change

Restrict the FOOOF frequency range and inspect how the table changes.

## Failure interpretation
If PSD works but FOOOF fails, you likely need the analysis extras.

## Next step
Continue to the TFR and connectivity worked examples, then read `11_advanced_utility_interoperability` if you need to combine PSD, baselining, and statistics helpers directly.

